In [1]:
print('hello world')

hello world


In [5]:
# install pdf reader
!pip install -q pypdf

# check the uploaded file is there
!ls -lh /content/*.pdf


-rw-r--r-- 1 root root 2.3M Oct 23 22:45 /content/ONDC_eB2B_Contract.pdf


In [6]:
# load the ONDC eB2B contract pdf in colab
from pypdf import PdfReader

pdf_path = "/content/ONDC_eB2B_Contract.pdf"
reader = PdfReader(pdf_path)

# read text page by page
pages = []
for i in range(len(reader.pages)):
    text = reader.pages[i].extract_text()
    if text is None:
        text = ""
    text = text.strip()
    if len(text) > 0:
        pages.append((i, text))

print("total pages with text:", len(pages))

# cut text into chunks, same setting as NITI run
chunks = []
for page_num, page_text in pages:
    start = 0
    while start < len(page_text):
        end = start + 1000
        part = page_text[start:end].strip()
        if len(part) > 200:
            chunk_id = "p" + str(page_num) + "_" + str(len(chunks))
            chunks.append({"id": chunk_id, "page": page_num, "text": part})
        start = start + 1000 - 150

print("total chunks:", len(chunks))
print("sample id:", chunks[5]["id"], "page:", chunks[5]["page"])
print(chunks[5]["text"][:300])


total pages with text: 97
total chunks: 328
sample id: p2_5 page: 2
"code":
 
"static_terms",
 
              
"value":
 
""
 
            
},
 
            
{
 
              
"code":
 
"static_terms_new
9
",
 
              
"value":
 
"https://github.com/ONDC-Official/NP-Static-Terms/buyerNP_BNP/1.0/tc.pdf"
 
            
},
 
            
{
 
              
"cod


In [7]:
# load embedding model
# same bi-encoder as NITI run for fair comparison
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# make list of all chunk texts
texts = []
for c in chunks:
    texts.append(c["text"])

# convert texts into vectors
vectors = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

print("vectors shape:", vectors.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

vectors shape: (328, 384)


In [10]:
!pip install -q faiss-cpu


In [11]:
# build search index using faiss
# normalized vectors so inner product equals cosine
import faiss
import numpy as np

lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
norm_vectors = vectors / lengths

size = norm_vectors.shape[1]
index = faiss.IndexFlatIP(size)
index.add(norm_vectors.astype("float32"))

print("total stored in index:", index.ntotal)


total stored in index: 328


In [12]:
# function to search top 10 chunks
def search_top10(query):
    # change query into numbers using same model
    q = model.encode([query], convert_to_numpy=True)
    # make it unit length for cosine
    q = q / np.linalg.norm(q, axis=1, keepdims=True)
    # search in index
    scores, ids = index.search(q.astype("float32"), 10)
    # collect results
    results = []
    for rank in range(10):
        idx = int(ids[0][rank])
        results.append({
            "rank": rank + 1,
            "id": chunks[idx]["id"],
            "page": chunks[idx]["page"],
            "score": float(scores[0][rank]),
            "text": chunks[idx]["text"]
        })
    return results

# test with a feature-allowedness style query
test_q = "Can a buyer app show catalog prices with taxes included"
hits = search_top10(test_q)
for h in hits:
    print("rank", h["rank"], "id", h["id"], "page", h["page"], "score", round(h["score"], 4))
    print(h["text"][:200])
    print("")


rank 1 id p45_167 page 45 score 0.406
the
 
retailer
 
is
 
identified.
 
Errors
 
50009,
 
50010,
 
50011,
 
50013,
 
50014;
 
30001,
 
40001.
 
Examples
 
/search
 
(Existing
 
Retailer
 
with
 
Customer
 
ID),
 
/search
 
(Existing
 
R

rank 2 id p82_282 page 82 score 0.3789
"descriptor":
 
{
 
              
"code":
 
"Serviceable
235
"
 
            
}
 
          
}
 
        
}
 
      
],
 
      
"quote":
 
{
 
        
"price":
 
{
 
          
"currency":
 
"INR",

rank 3 id p46_168 page 46 score 0.3315
retailer_info_required
 
that
 
applies
 
(the
 
provider’s
 
own,
 
else
 
the
 
descriptor’s)
 
Who
 
may
 
be
 
shown
 
the
 
catalogue
 
A
 
variant
 
with
 
type
 
=
 
new
 
is
 
published
 
Ever

rank 4 id p88_302 page 88 score 0.3177
1",
 
            
"title":
 
"Delivery
 
charges",
 
            
"@ondc/org/title_type":
 
"delivery",
 
            
"price":
 
{
 
              
"currency":
 
"INR",
 
              
"value":
 
"

rank 5 id p52_187 page 52 score 0.310

In [14]:
!pip install -q google-generativeai


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.16.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bi

In [15]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
llm = genai.GenerativeModel("models/gemini-3.8-flash")
print(llm.generate_content("say ok").text)


/tmp/ipykernel_2026/938914743.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


OK.


In [16]:
# fixed prompt for both pipelines
# only chunks change, prompt stays same
def make_prompt(query, top3):
    context = ""
    for i in range(3):
        context = context + "\nChunk " + str(i+1) + ":\n" + top3[i]["text"] + "\n"
    prompt = "You are a helper for the ONDC eB2B protocol document.\n"
    prompt = prompt + "Answer only from given context.\n"
    prompt = prompt + "If answer is not in context, say not found.\n"
    prompt = prompt + "\nContext:\n" + context + "\nQuestion: " + query + "\nAnswer:"
    return prompt

def get_answer1(query):
    hits = search_top10(query)
    top3 = hits[0:3]
    prompt = make_prompt(query, top3)
    out = llm.generate_content(prompt)
    return top3, out.text

# test - buyer app feature allowedness question
q = "Can a buyer app show catalog prices with taxes included"
top3, ans1 = get_answer1(q)
print(ans1)
print("")
print("top3 ids used:")
for h in top3:
    print(h["rank"], h["id"], "page", h["page"])


Based on the context, the **item price should be either inclusive or exclusive of taxes, exactly as defined in the catalog**. 

*(Note: Specific display rules for a "buyer app" are not found in the context).*

top3 ids used:
1 p45_167 page 45
2 p82_282 page 82
3 p46_168 page 46


In [17]:
# load cross encoder model
# this model reads query and chunk together
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# test with one pair
test_score = reranker.predict([["Can a buyer app show prices with taxes", "price may be inclusive or exclusive of taxes as defined in catalog"]])
print(test_score)
print("model loaded ok")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[-4.6882005]
model loaded ok


In [18]:
# function to re rank top 10 using cross encoder
def rerank_top10(query, hits):
    # make pairs of query and chunk text
    pairs = []
    for h in hits:
        pairs.append([query, h["text"]])
    # get scores from cross encoder
    scores = reranker.predict(pairs)
    # add new score to each hit
    for i in range(len(hits)):
        hits[i]["cross_score"] = float(scores[i])
    # sort by new score, high to low
    ranked = sorted(hits, key=lambda x: x["cross_score"], reverse=True)
    return ranked

def get_answer2(query):
    hits = search_top10(query)
    ranked = rerank_top10(query, hits)
    top3_new = ranked[0:3]
    prompt = make_prompt(query, top3_new)
    out = llm.generate_content(prompt)
    return hits, ranked, top3_new, out.text

# same question, compare pipelines
q = "Can a buyer app show catalog prices with taxes included"
old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
for new_pos in range(3):
    c = new_ranked[new_pos]
    old_rank = 0
    for h in old_hits:
        if h["id"] == c["id"]:
            old_rank = h["rank"]
    print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
print("Answer 2:")
print(ans2)


chunk p45_167 page 45 old 1 new 1
chunk p43_158 page 43 old 10 new 2
chunk p46_168 page 46 old 3 new 3
Answer 2:
Not found.


In [19]:
# three candidate failure queries for ONDC buyer vs seller apps
queries = [
    "Can a seller app cancel an order after it is confirmed by the buyer",
    "Is a buyer app allowed to charge extra delivery fee over the seller quoted price",
    "What fulfillment status must a seller app publish for buyer tracking"
]

for q in queries:
    print("Question:", q)
    hits = search_top10(q)
    ranked = rerank_top10(q, hits)
    for new_pos in range(3):
        c = ranked[new_pos]
        old_rank = 0
        for h in hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    print("")


Question: Can a seller app cancel an order after it is confirmed by the buyer
chunk p53_193 page 53 old 2 new 1
chunk p60_210 page 60 old 4 new 2
chunk p57_200 page 57 old 1 new 3

Question: Is a buyer app allowed to charge extra delivery fee over the seller quoted price
chunk p95_324 page 95 old 1 new 1
chunk p82_283 page 82 old 6 new 2
chunk p57_201 page 57 old 9 new 3

Question: What fulfillment status must a seller app publish for buyer tracking
chunk p53_192 page 53 old 4 new 1
chunk p56_198 page 56 old 2 new 2
chunk p4_16 page 4 old 3 new 3



In [20]:
# answers for the two best queries
test_queries = [
    "Is a buyer app allowed to charge extra delivery fee over the seller quoted price",
    "What fulfillment status must a seller app publish for buyer tracking"
]

for q in test_queries:
    print("Question:", q)
    top3_old, ans1 = get_answer1(q)
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    print("Answer 1:")
    print(ans1[:700])
    print("")
    print("Answer 2:")
    print(ans2[:700])
    print("")
    print("----")
    print("")


Question: Is a buyer app allowed to charge extra delivery fee over the seller quoted price
Answer 1:
Not found

Answer 2:
Not found

----

Question: What fulfillment status must a seller app publish for buyer tracking
Answer 1:
Not found

Answer 2:
Not found.

----



In [21]:
# show full text of the promoted chunks
show_ids = ["p82_283", "p57_201", "p53_192", "p56_198", "p60_210", "p95_324"]
seen = {}
hits = search_top10("delivery fee fulfillment status cancel order")
ranked = rerank_top10("delivery fee fulfillment status cancel order", hits)
for c in ranked:
    seen[c["id"]] = c
for cid in show_ids:
    if cid in seen:
        c = seen[cid]
        print("chunk", c["id"], "page", c["page"])
        print(c["text"])
        print("")
        print("----")
        print("")


chunk p82_283 page 82
level
 
"item"
 
(product
 
cost),
 
"tax",
 
"discount";
 
fulfillment
 
level
 
"delivery"
 
(shipping
 
charges),
 
"packing",
 
"misc"
 
(convenience
 
fees),
 
"tax"
 
(with
 
quote
 
type
 
"fulfillment");
 
order
 
level
 
as
 
per
 
offer
 
construct;
 
title
 
text
 
can
 
be
 
customized
 
by
 
BNP;
 
235
 
enum
 
-
 
"Serviceable",
 
"Non-serviceable";
 
234
 
SNP
 
performs
 
serviceability
 
checks
 
based
 
on
 
end
 
location
 
in
 
/select.
 
If
 
non
 
serviceable,
 
SNP
 
has
 
to
 
return
 
the
 
appropriate
 
error
 
code
 
along
 
with
 
a
 
valid
 
/on_select
 
response;
 
233
 
Optional;
 
required
 
if
 
already
 
existing
 
customer_id
 
provided
 
in
 
/select
 
 
232
 
Order-to-delivery
 
(O2D)
 
TAT
 
shown
 
=
 
max(time_to_ship
 
for
 
items
 
in
 
the
 
fulfillment)
 
+
 
ship-to-delivery
 
(S2D)
 
time
 
provided
 
by
 
LSP
 
in
 
TAT;
 
231
 
maps
 
to
 
logistics
 
category
 
as
 
defined
 
in
 
Logistics
 
API
 
contract;
 
230
 

In [22]:
# show full text of key chunks by id
want = ["p82_283", "p57_201", "p53_192", "p56_198", "p60_210", "p95_324", "p43_158"]
for c in chunks:
    if c["id"] in want:
        print("chunk", c["id"], "page", c["page"])
        print(c["text"])
        print("")
        print("----")
        print("")


chunk p43_158 page 43
}
 
}
 
 
N.B.
 
The
 
mapping
 
carries
 
no
 
catalogue,
 
prices,
 
credit
 
or
 
offers.
 
Each
 
distributor
 
returns
 
those
 
in
 
/on_search
 
(Existing
 
Retailer)
.
 
 
Response
 
to
 
the
 
one-retailer
 
lookup:
 
{
 
  
"context":
 
{
 
    
"domain":
 
"ONDC:RETeB2B",
 
    
"country":
 
"IND",
 
    
"city":
 
"std:080",
 
    
"action":
 
"on_search",
 
    
"core_version":
 
"1.2.5",
 
    
"bap_id":
 
"buyer-np.com",
 
    
"bap_uri":
 
"https://buyer-np.com/ondc",
 
    
"bpp_id":
 
"seller-np.com",
 
    
"bpp_uri":
 
"https://seller-np.com/ondc",
 
    
"transaction_id":
 
"T4",
 
    
"message_id":
 
"M1",
 
    
"timestamp":
 
"2025-01-08T08:00:01.000Z"
 
  
},
 
  
"message":
 
{
 
    
"catalog":
 
{
 
      
"bpp/descriptor":
 
{
 
        
"tags":
 
[
 
          
{
 
            
"code":
 
"retailer_mapping",
 
            
"list":
 
[
 
              
{
 
                
"code":
 
"customer_id",
 
                
"value":
 
"CUST-20

In [24]:
# rewritten queries using the document's own terms
queries = [
    "Can a buyer app add convenience fees to an order",
    "Can a new retailer place an order before distributor mapping is complete",
    "What happens if provider id is omitted without descriptor type new"
]

for q in queries:
    print("Question:", q)
    hits = search_top10(q)
    ranked = rerank_top10(q, hits)
    for new_pos in range(3):
        c = ranked[new_pos]
        old_rank = 0
        for h in hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    print("")


Question: Can a buyer app add convenience fees to an order
chunk p82_283 page 82 old 8 new 1
chunk p95_324 page 95 old 2 new 2
chunk p37_140 page 37 old 5 new 3

Question: Can a new retailer place an order before distributor mapping is complete
chunk p53_192 page 53 old 2 new 1
chunk p3_9 page 3 old 4 new 2
chunk p38_144 page 38 old 1 new 3

Question: What happens if provider id is omitted without descriptor type new
chunk p61_216 page 61 old 6 new 1
chunk p53_191 page 53 old 1 new 2
chunk p60_209 page 60 old 7 new 3



In [25]:
# answers for all three failure queries - run once only
final_queries = [
    "Can a buyer app add convenience fees to an order",
    "Can a new retailer place an order before distributor mapping is complete",
    "What happens if provider id is omitted without descriptor type new"
]

for q in final_queries:
    print("Question:", q)
    top3_old, ans1 = get_answer1(q)
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    print("Answer 1:")
    print(ans1[:700])
    print("")
    print("Answer 2:")
    print(ans2[:700])
    print("")
    print("----")
    print("")


Question: Can a buyer app add convenience fees to an order
Answer 1:
Not found.

Answer 2:
not found

----

Question: Can a new retailer place an order before distributor mapping is complete
Answer 1:
Yes. If the assigned distributor's (P1's) applicable group has `type = new`, the retailer can already order while creation is pending, with its details in the customer object (before the retailer is officially mapped and identified via the unsolicited `/on_search`).

Answer 2:
Based on the provided context, yes: when the Seller NP picks a distributor and answers with "creation pending" (naming it, e.g., P1), the retailer can already order if P1’s applicable group has `type = new`.

----

Question: What happens if provider id is omitted without descriptor type new
Answer 1:
Based on the provided context, leaving out `provider.id` without a descriptor `type = new` gets a 30000 NACK.

Answer 2:
Based on the provided context, leaving out `provider.id` without a descriptor `type = new` results

In [27]:
# switch to lite model with fresh quota
# same prompt functions, only the model object changes
llm = genai.GenerativeModel("models/gemini-flash-lite-latest")
print(llm.generate_content("say ok").text)


ok


In [28]:
# rewritten queries, print each answer right away
retry_queries = [
    "What misc fees appear at fulfillment level in a quote and who customizes their title text",
    "What error codes are returned when retailer creation lapses or is declined"
]

for q in retry_queries:
    print("Question:", q)
    top3_old, ans1 = get_answer1(q)
    print("baseline top3 ids:", [h["id"] for h in top3_old])
    print("Answer 1:")
    print(ans1[:600])
    print("")
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    print("reranked top3 ids:", [h["id"] for h in new_top3])
    for new_pos in range(3):
        c = new_ranked[new_pos]
        old_rank = 0
        for h in old_hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    print("Answer 2:")
    print(ans2[:600])
    print("")
    print("----")
    print("")


Question: What misc fees appear at fulfillment level in a quote and who customizes their title text
baseline top3 ids: ['p78_271', 'p82_282', 'p56_198']
Answer 1:
Based on the provided context, "misc" is listed as a fulfillment level type (for convenience, though the text cuts off). However, the context does not mention anything about who customizes their title text. 

Therefore, the full answer is not found.

reranked top3 ids: ['p82_283', 'p78_271', 'p82_282']
chunk p82_283 page 82 old 4 new 1
chunk p78_271 page 78 old 1 new 2
chunk p82_282 page 82 old 2 new 3
Answer 2:
Based on the provided context, "misc" (convenience fees) appears at the fulfillment level. The title text can be customized by BNP.

----

Question: What error codes are returned when retailer creation lapses or is declined
baseline top3 ids: ['p64_232', 'p59_208', 'p65_233']
Answer 1:
Based on the provided context, the error codes are:
- **50015**: When the Seller NP declines retailer creation.
- **50016**: When the 

In [29]:
# dump candidate gold chunks to design the last scoring query
want = ["p37_140", "p78_271", "p59_208", "p63_225", "p64_230", "p3_9", "p38_144"]
for c in chunks:
    if c["id"] in want:
        print("chunk", c["id"], "page", c["page"])
        print(c["text"])
        print("")
        print("----")
        print("")


chunk p3_9 page 3
-
 
retailer_mapping_required
 
lists
 
the
 
codes
 
that
 
find
 
which
 
distributors
 
serve
 
a
 
retailer.
 
It
 
is
 
published
 
at
 
bpp/descriptor.tags.
 
-
 
No
 
retailer-specific
 
data,
 
such
 
as
 
credit
 
limits,
 
scheme
 
progress
 
or
 
POC.
 
-
 
The
 
full
 
catalogue
 
arrives
 
before
 
any
 
retailer
 
is
 
identified.
 
The
 
Buyer
 
NP
 
caches
 
it
 
per
 
provider.
 
 
What
 
the
 
Buyer
 
NP
 
shows
 
a
 
retailer
 
-
 
retailer_mapping_required
 
decides
 
which
 
distributors
 
the
 
retailer
 
sees.
 
See
 
Retailer
 
Mapping
.
 
-
 
retailer_info_required
 
decides
 
whether
 
each
 
distributor’s
 
catalogue
 
opens
 
before
 
the
 
retailer
 
is
 
identified.
 
See
 
Retailer
 
Identification
 
&
 
Terms
.
 
 
 
This
 
example
 
publishes
 
both
 
declarations
 
at
 
bpp/descriptor.tags.
 
P1
 
also
 
publishes
 
its
 
own
 
retailer_info_required,
 
so
 
its
 
own
 
group
 
applies
 
to
 
P1.
 
P2
 
publishes
 
none,
 
so
 
the
 


In [30]:
# second scoring case - distributor assignment rule
q = "Can a retailer buy from a different distributor than the one assigned by the brand"

top3_old, ans1 = get_answer1(q)
print("baseline top3 ids:", [h["id"] for h in top3_old])
print("Answer 1:")
print(ans1[:600])
print("")
old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
print("reranked top3 ids:", [h["id"] for h in new_top3])
for new_pos in range(3):
    c = new_ranked[new_pos]
    old_rank = 0
    for h in old_hits:
        if h["id"] == c["id"]:
            old_rank = h["rank"]
    print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
print("Answer 2:")
print(ans2[:600])


baseline top3 ids: ['p36_133', 'p38_144', 'p45_167']
Answer 1:
No. According to the context, the retailer must buy from that distributor because its credit, schemes, and claims sit there, and orders to any other are rejected.

reranked top3 ids: ['p38_144', 'p36_133', 'p36_134']
chunk p38_144 page 38 old 2 new 1
chunk p36_133 page 36 old 1 new 2
chunk p36_134 page 36 old 4 new 3
Answer 2:
No, the retailer must buy from that distributor; its credit, schemes and claims sit there, and orders to any other are rejected.


In [31]:
# last two candidates - ranks only
queries = [
    "Which declaration decides which distributors a retailer is shown",
    "What feature codes does a buyer app declare for retailer identification and mapping"
]

for q in queries:
    print("Question:", q)
    hits = search_top10(q)
    ranked = rerank_top10(q, hits)
    print("baseline top3 ids:", [h["id"] for h in hits[:3]])
    for new_pos in range(3):
        c = ranked[new_pos]
        old_rank = 0
        for h in hits:
            if h["id"] == c["id"]:
                old_rank = h["rank"]
        print("chunk", c["id"], "page", c["page"], "old", old_rank, "new", new_pos+1)
    print("")



Question: Which declaration decides which distributors a retailer is shown
baseline top3 ids: ['p36_133', 'p38_144', 'p45_167']
chunk p3_9 page 3 old 4 new 1
chunk p38_144 page 38 old 2 new 2
chunk p37_138 page 37 old 5 new 3

Question: What feature codes does a buyer app declare for retailer identification and mapping
baseline top3 ids: ['p38_142', 'p45_165', 'p36_132']
chunk p37_140 page 37 old 4 new 1
chunk p2_7 page 2 old 6 new 2
chunk p38_142 page 38 old 1 new 3



In [32]:
# final answers for the two qualifying candidates - run once
last_queries = [
    "Which declaration decides which distributors a retailer is shown",
    "What feature codes does a buyer app declare for retailer identification and mapping"
]

for q in last_queries:
    print("Question:", q)
    top3_old, ans1 = get_answer1(q)
    print("Answer 1:")
    print(ans1[:600])
    print("")
    old_hits, new_ranked, new_top3, ans2 = get_answer2(q)
    print("Answer 2:")
    print(ans2[:600])
    print("")
    print("----")
    print("")


Question: Which declaration decides which distributors a retailer is shown
Answer 1:
Based on the provided context, `retailer_mapping_required`, published at `bpp/descriptor.tags`, decides which providers (distributors) a retailer sees.

Answer 2:
retailer_mapping_required

----

Question: What feature codes does a buyer app declare for retailer identification and mapping
Answer 1:
Based on the provided context:
- Retailer Identification & Terms uses feature code **030** in `bap_features`.
- Retailer Mapping uses feature code **031** in `bap_features` of the broadcast `/search` (which implies 030).

Answer 2:
Based on the provided context, the Buyer NP declares feature code **030** for retailer identification and **031** for retailer mapping.

----



In [33]:
# latency: retrieval averages plus one end-to-end timing
import time

q = "What misc fees appear at fulfillment level in a quote and who customizes their title text"

# retrieval only, no LLM, average of 5
t1 = time.time()
for i in range(5):
    search_top10(q)
t2 = time.time()
base_time = (t2 - t1) / 5

hits = search_top10(q)
t1 = time.time()
for i in range(5):
    rerank_top10(q, hits)
t2 = time.time()
rerank_time = (t2 - t1) / 5

print("baseline search time:", round(base_time, 4))
print("rerank extra time:", round(rerank_time, 4))

# end to end once each
t1 = time.time()
get_answer1(q)
t2 = time.time()
print("Pipeline 1 with LLM:", round(t2 - t1, 3))
t1 = time.time()
get_answer2(q)
t2 = time.time()
print("Pipeline 2 with LLM:", round(t2 - t1, 3))


baseline search time: 0.0075
rerank extra time: 0.0437
Pipeline 1 with LLM: 0.951
Pipeline 2 with LLM: 0.732
